# Projet IA — Analyseur de Scènes : Conduite de Nuit
## Notebook 1 : Exploration + Entraînement + Évaluation
**Module IA 2025-2026 — YOLOv8n vs YOLOv8s sur BDD100K (nuit)**

1. Exploration du dataset BDD100K (nuit)
2. Conversion des labels en format YOLO
3. Entraînement de 2 modèles (YOLOv8n vs YOLOv8s)
4. Évaluation et comparaison (mAP, précision, rappel)
5. Visualisation des détections



## Installation & Connexion Google Drive

In [1]:
# Installation des librairies nécessaires
!pip install ultralytics -q
!pip install groq -q
print("✅ Installation terminée")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.7 MB/s eta 0:00:00
✅ Installation terminée


In [3]:
# Connexion à Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive connecté")


Mounted at /content/drive
✅ Google Drive connecté


In [4]:
import os, json, random, shutil, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO

# ───────────────────────────────────────────────────────────────────────────────

# Les chemins des donnes de tests des bdd100k
BDD_JSON   = '/content/drive/MyDrive/bdd100k/labels/bdd100k_labels_images_train.json'
BDD_IMAGES = '/content/drive/MyDrive/bdd100k/images/100k/train'

# Dossier de travail local
WORK_DIR   = '/content/conduite_nuit'
DATA_YAML  = '/content/data.yaml'

# ───────────────────────────────────────────────────────────────────────────────

print(" Chemins configurés")
print(f"  BDD JSON   : {BDD_JSON}")
print(f"  BDD Images : {BDD_IMAGES}")
print(f"  Work dir   : {WORK_DIR}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Chemins configurés
  BDD JSON   : /content/drive/MyDrive/bdd100k/labels/bdd100k_labels_images_train.json
  BDD Images : /content/drive/MyDrive/bdd100k/images/100k/train
  Work dir   : /content/conduite_nuit


## Exploration du Dataset BDD100K

In [5]:
# Chargement du JSON BDD100K
print("📖 Chargement du JSON BDD100K...")
with open(BDD_JSON, 'r') as f:
    bdd_data = json.load(f)
print(f"✅ {len(bdd_data)} images dans BDD100K train")

# On compte les conditions météo et heures de la journée
from collections import Counter

heures   = Counter()
meteos   = Counter()
classes  = Counter()

for entry in bdd_data:
    attr = entry.get('attributes', {})
    heures[attr.get('timeofday', 'unknown')] += 1
    meteos[attr.get('weather', 'unknown')]   += 1
    for label in entry.get('labels', []):
        classes[label['category']] += 1

print("\n🕐 Répartition par heure de la journée :")
for k, v in sorted(heures.items(), key=lambda x: -x[1]):
    print(f"   {k:15s} : {v:6d} images")

print("\n🌦️  Répartition météo :")
for k, v in sorted(meteos.items(), key=lambda x: -x[1]):
    print(f"   {k:15s} : {v:6d} images")

print("\n📦 Classes d'objets (toutes images) :")
for k, v in classes.most_common(15):
    print(f"   {k:20s} : {v:7d} annotations")


📖 Chargement du JSON BDD100K...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/bdd100k/labels/bdd100k_labels_images_train.json'

In [ ]:
# Graphiques d'exploration
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Exploration BDD100K — Distribution des données", fontsize=14, fontweight='bold')

# 1. Heure de la journée
axes[0].bar(heures.keys(), heures.values(), color=['#2c3e50','#f39c12','#3498db','#27ae60'])
axes[0].set_title("Heure de la journée")
axes[0].set_ylabel("Nombre d'images")
axes[0].tick_params(axis='x', rotation=30)

# 2. Météo
axes[1].bar(meteos.keys(), meteos.values(), color='#3498db', alpha=0.8)
axes[1].set_title("Conditions météo")
axes[1].tick_params(axis='x', rotation=30)

# 3. Top 8 classes
top_classes = dict(classes.most_common(8))
axes[2].barh(list(top_classes.keys()), list(top_classes.values()), color='#e74c3c', alpha=0.8)
axes[2].set_title("Top 8 classes d'objets")
axes[2].set_xlabel("Nombre d'annotations")

plt.tight_layout()
plt.savefig('/content/exploration_bdd100k.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Graphique sauvegardé")


In [ ]:
# ─── Focus sur les images nocturnes ─────────────────────────────────────────
# On garde : night + dawn/dusk, et seulement les classes qui nous intéressent
CLASS_MAP = {
    'pedestrian'  : 0,
    'car'         : 1,
    'truck'       : 2,
    'bus'         : 3,
    'traffic sign': 4,
    'traffic light': 5,
}

images_nuit = []
for entry in bdd_data:
    attr  = entry.get('attributes', {})
    heure = attr.get('timeofday', '')
    if heure not in ['night', 'dawn/dusk']:
        continue
    cats = [l['category'] for l in entry.get('labels', [])]
    if not any(c in CLASS_MAP for c in cats):
        continue
    img_path = os.path.join(BDD_IMAGES, entry['name'])
    if not os.path.exists(img_path):
        continue
    images_nuit.append(entry)

print(f"🌙 Images nocturnes avec nos classes : {len(images_nuit)}")
print(f"   (sur {heures.get('night', 0) + heures.get('dawn/dusk', 0)} images nocturnes totales)")

# Distribution des classes dans les images nocturnes
classes_nuit = Counter()
for entry in images_nuit:
    for label in entry.get('labels', []):
        if label['category'] in CLASS_MAP:
            classes_nuit[label['category']] += 1

print("\n📦 Classes dans les images nocturnes :")
for cls, count in sorted(classes_nuit.items(), key=lambda x: -x[1]):
    print(f"   {cls:20s} : {count:6d} annotations")


## Conversion BDD100K → Format YOLO

In [ ]:
# Conversion du dataset au format YOLO
# (bounding boxes normalisées : x_centre, y_centre, largeur, hauteur)

IMG_W, IMG_H = 1280, 720

def bbox_to_yolo(x1, y1, x2, y2):
    """Convertit une bbox absolue en coordonnées YOLO normalisées [0,1]."""
    xc = ((x1 + x2) / 2) / IMG_W
    yc = ((y1 + y2) / 2) / IMG_H
    w  = (x2 - x1) / IMG_W
    h  = (y2 - y1) / IMG_H
    return xc, yc, w, h

def convertir_dataset(images_nuit, bdd_images_dir, work_dir, max_images=600, ratio_val=0.2):
    """Filtre et convertit les images BDD100K en format YOLO."""
    random.seed(42)

    # Création des dossiers
    for split in ['train', 'val']:
        os.makedirs(f'{work_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{work_dir}/labels/{split}', exist_ok=True)
    os.makedirs(f'{work_dir}/sample_images', exist_ok=True)

    # Sélection et mélange
    selection = images_nuit[:max_images]
    random.shuffle(selection)
    nb_val    = int(len(selection) * ratio_val)
    val_names = {e['name'] for e in selection[:nb_val]}

    compteur = {'train': 0, 'val': 0}
    nb_demo  = 0

    for entry in selection:
        nom   = entry['name']
        split = 'val' if nom in val_names else 'train'
        src   = os.path.join(bdd_images_dir, nom)

        # Copie image
        shutil.copy(src, f'{work_dir}/images/{split}/{nom}')

        # Génération label YOLO
        lignes = []
        for label in entry.get('labels', []):
            cat = label['category']
            if cat not in CLASS_MAP or 'box2d' not in label:
                continue
            b  = label['box2d']
            x1 = max(0, min(IMG_W, b['x1']))
            y1 = max(0, min(IMG_H, b['y1']))
            x2 = max(0, min(IMG_W, b['x2']))
            y2 = max(0, min(IMG_H, b['y2']))
            if x2 <= x1 or y2 <= y1:
                continue
            xc, yc, w, h = bbox_to_yolo(x1, y1, x2, y2)
            lignes.append(f"{CLASS_MAP[cat]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        nom_label = nom.replace('.jpg', '.txt')
        with open(f'{work_dir}/labels/{split}/{nom_label}', 'w') as f:
            f.write('\n'.join(lignes))

        compteur[split] += 1

        # Quelques images pour la démo
        if split == 'train' and nb_demo < 8:
            shutil.copy(src, f'{work_dir}/sample_images/{nom}')
            nb_demo += 1

    return compteur

print("🔄 Conversion en cours...")
compteur = convertir_dataset(images_nuit, BDD_IMAGES, WORK_DIR, max_images=600)
print(f"✅ Conversion terminée !")
print(f"   Train : {compteur['train']} images")
print(f"   Val   : {compteur['val']} images")


In [ ]:
# Génération du fichier data.yaml (configuration YOLO)
yaml_content = f"""path: {WORK_DIR}
train: images/train
val:   images/val

nc: {len(CLASS_MAP)}
names: {list(CLASS_MAP.keys())}
"""

with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml créé :")
print(yaml_content)


In [ ]:
# ─── Visualisation d'images nocturnes avec leurs bounding boxes ───────────────
# On affiche 6 images pour vérifier que les labels sont corrects

COULEURS = {
    0: '#e74c3c',  # pedestrian — rouge
    1: '#3498db',  # car — bleu
    2: '#27ae60',  # truck — vert
    3: '#f39c12',  # bus — orange
    4: '#9b59b6',  # traffic sign — violet
    5: '#1abc9c',  # traffic light — turquoise
}
NOMS = list(CLASS_MAP.keys())

train_imgs = glob.glob(f'{WORK_DIR}/images/train/*.jpg')
random.shuffle(train_imgs)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle("Exemples d'images nocturnes avec labels YOLO", fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flat, train_imgs[:6]):
    img   = mpimg.imread(img_path)
    h, w  = img.shape[:2]
    label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')

    ax.imshow(img)
    ax.axis('off')
    ax.set_title(Path(img_path).name[:25], fontsize=8)

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                x = (xc - bw/2) * w
                y = (yc - bh/2) * h
                rect = patches.Rectangle(
                    (x, y), bw*w, bh*h,
                    linewidth=2, edgecolor=COULEURS.get(cls, 'white'), facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x, y-4, NOMS[cls], fontsize=7, color=COULEURS.get(cls, 'white'),
                        bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.6))

plt.tight_layout()
plt.savefig('/content/visualisation_labels.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Visualisation sauvegardée")


## Entraînement YOLOv8n

In [ ]:
# ─── ENTRAÎNEMENT YOLOv8n ────────────────────────────────────────────────────
# YOLOv8n = nano, très rapide
# Augmentations adaptées à la nuit : luminosité réduite, flou, bruit

model_n = YOLO('yolov8n.pt')  # pré-entraîné sur COCO (80 classes)

print(" Lancement de l'entraînement YOLOv8n...")
print("   Durée estimée : 20-30 min sur GPU T4\n")

results_n = model_n.train(
    data       = DATA_YAML,
    epochs     = 30,          # 30 epochs suffisent pour un fine-tuning
    imgsz      = 640,         # taille d'image standard YOLO
    batch      = 16,          # adapter si out of memory → mettre 8
    name       = 'conduite_nuit_yolov8n',
    project    = '/content/runs/train',
    # ── Augmentations adaptées aux conditions nocturnes ──
    hsv_v      = 0.6,         # variation de luminosité (crucial pour la nuit !)
    hsv_s      = 0.5,         # variation de saturation
    flipud     = 0.0,         # pas de flip vertical (images de dashcam)
    fliplr     = 0.5,         # flip horizontal (bonne augmentation)
    mosaic     = 1.0,         # composition mosaïque (4 images en 1)
    mixup      = 0.15,        # mélange d'images pour la robustesse
    degrees    = 0.0,         # pas de rotation (images de route)
    # ── Paramètres d'optimisation ──
    optimizer  = 'AdamW',
    lr0        = 0.001,
    patience   = 10,          # early stopping si pas d'amélioration
    verbose    = True,
    plots      = True,
)

print("\n✅ Entraînement YOLOv8n terminé !")


## Étape 4 — Entraînement YOLOv8s (modèle plus précis)

In [ ]:
# ─── ENTRAÎNEMENT YOLOv8s ────────────────────────────────────────────────────
# YOLOv8s = small, plus précis que nano

model_s = YOLO('yolov8s.pt')

print(" Lancement de l'entraînement YOLOv8s...")
print("   Durée estimée : 30-45 min sur GPU T4\n")

results_s = model_s.train(
    data       = DATA_YAML,
    epochs     = 30,
    imgsz      = 640,
    batch      = 16,
    name       = 'conduite_nuit_yolov8s',
    project    = '/content/runs/train',
    # Mêmes augmentations nocturnes
    hsv_v      = 0.6,
    hsv_s      = 0.5,
    flipud     = 0.0,
    fliplr     = 0.5,
    mosaic     = 1.0,
    mixup      = 0.15,
    degrees    = 0.0,
    optimizer  = 'AdamW',
    lr0        = 0.001,
    patience   = 10,
    verbose    = True,
    plots      = True,
)

print("\n✅ Entraînement YOLOv8s terminé !")


## Évaluation et Comparaison des Modèles

In [ ]:
# ─── ÉVALUATION DES DEUX MODÈLES ─────────────────────────────────────────────
# On évalue sur le jeu de validation (images non vues pendant l'entraînement)

def evaluer_modele(nom, chemin_poids, data_yaml):
    """Évalue un modèle YOLO et retourne ses métriques principales."""
    print(f"📏 Évaluation de {nom}...")
    model   = YOLO(chemin_poids)
    metrics = model.val(data=data_yaml, verbose=False)

    return {
        'Modèle'       : nom,
        'mAP@0.5'      : round(metrics.box.map50,   4),
        'mAP@0.5:0.95' : round(metrics.box.map,     4),
        'Précision'    : round(metrics.box.mp,       4),
        'Rappel'       : round(metrics.box.mr,       4),
    }

# Chemins des meilleurs poids
chemin_n = '/content/runs/train/conduite_nuit_yolov8n/weights/best.pt'
chemin_s = '/content/runs/train/conduite_nuit_yolov8s/weights/best.pt'

resultats = []
resultats.append(evaluer_modele('YOLOv8n (nano)', chemin_n, DATA_YAML))
resultats.append(evaluer_modele('YOLOv8s (small)', chemin_s, DATA_YAML))

df = pd.DataFrame(resultats).set_index('Modèle')
print("\n" + "="*60)
print("📊 TABLEAU COMPARATIF DES PERFORMANCES")
print("="*60)
print(df.to_string())
print("="*60)


In [ ]:
# ─── GRAPHIQUE DE COMPARAISON ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Comparaison YOLOv8n vs YOLOv8s — Conduite de Nuit", fontsize=14, fontweight='bold')

metriques = ['mAP@0.5', 'mAP@0.5:0.95', 'Précision', 'Rappel']
couleurs  = ['#3498db', '#e74c3c']

# Barres groupées
x     = range(len(metriques))
width = 0.35

for i, (nom, row) in enumerate(df.iterrows()):
    valeurs = [row[m] for m in metriques]
    offset  = (i - 0.5) * width
    bars    = axes[0].bar([xi + offset for xi in x], valeurs, width,
                          label=nom, color=couleurs[i], alpha=0.85)
    # Valeurs sur les barres
    for bar, val in zip(bars, valeurs):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[0].set_xticks(list(x))
axes[0].set_xticklabels(metriques, fontsize=10)
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.1)
axes[0].legend()
axes[0].set_title("Métriques de performance")
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Seuil 50%')

# Radar / spider chart simplifié — tableau récap
axes[1].axis('off')
table_data = [[m] + [f"{row[m]:.4f}" for _, row in df.iterrows()] for m in metriques]
table = axes[1].table(
    cellText   = table_data,
    colLabels  = ['Métrique'] + list(df.index),
    cellLoc    = 'center',
    loc        = 'center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)
# Couleur d'en-tête
for j in range(len(df.index) + 1):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
axes[1].set_title("Tableau récapitulatif")

plt.tight_layout()
plt.savefig('/content/comparaison_modeles.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Graphique de comparaison sauvegardé")


In [ ]:
# ─── COURBES D'ENTRAÎNEMENT ───────────────────────────────────────────────────
# On trace les courbes loss et mAP par epoch pour les deux modèles

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Courbes d'entraînement — YOLOv8n vs YOLOv8s", fontsize=14, fontweight='bold')

configs = [
    ('YOLOv8n', '/content/runs/train/conduite_nuit_yolov8n/results.csv', '#3498db'),
    ('YOLOv8s', '/content/runs/train/conduite_nuit_yolov8s/results.csv', '#e74c3c'),
]

for nom, csv_path, couleur in configs:
    if not os.path.exists(csv_path):
        print(f"⚠️ Fichier manquant : {csv_path}")
        continue

    df_r = pd.read_csv(csv_path)
    df_r.columns = [c.strip() for c in df_r.columns]  # nettoyer les espaces

    epochs = df_r['epoch']

    # Loss d'entraînement (box + cls + dfl)
    try:
        loss_train = df_r['train/box_loss'] + df_r['train/cls_loss']
        axes[0, 0].plot(epochs, loss_train, label=nom, color=couleur, linewidth=2)
    except KeyError:
        pass

    # Loss de validation
    try:
        loss_val = df_r['val/box_loss'] + df_r['val/cls_loss']
        axes[0, 1].plot(epochs, loss_val, label=nom, color=couleur, linewidth=2)
    except KeyError:
        pass

    # mAP@0.5
    try:
        axes[1, 0].plot(epochs, df_r['metrics/mAP50(B)'], label=nom, color=couleur, linewidth=2)
    except KeyError:
        pass

    # mAP@0.5:0.95
    try:
        axes[1, 1].plot(epochs, df_r['metrics/mAP50-95(B)'], label=nom, color=couleur, linewidth=2)
    except KeyError:
        pass

for ax, titre, ylabel in zip(
    axes.flat,
    ['Loss (train)', 'Loss (validation)', 'mAP@0.5', 'mAP@0.5:0.95'],
    ['Loss', 'Loss', 'mAP', 'mAP']
):
    ax.set_title(titre, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/courbes_entrainement.png', dpi=120, bbox_inches='tight')
plt.show()


##Visualisation des Détections

In [ ]:
# ─── PRÉDICTIONS VISUELLES SUR IMAGES NOCTURNES ───────────────────────────────
# On utilise le meilleur modèle (YOLOv8s) pour faire des prédictions

best_model = YOLO(chemin_s)  # YOLOv8s = plus précis

# Images de validation = images non vues pendant l'entraînement
val_images = glob.glob(f'{WORK_DIR}/images/val/*.jpg')
random.shuffle(val_images)
sample_images = val_images[:6]

print(f"🔍 Prédiction sur {len(sample_images)} images de validation...")

COULEURS_RGB = {
    'pedestrian'  : (231, 76,  60),
    'car'         : (52,  152, 219),
    'truck'       : (46,  204, 113),
    'bus'         : (243, 156, 18),
    'traffic sign': (155, 89,  182),
    'traffic light': (26, 188, 156),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Détections YOLOv8s — Conduite de Nuit (images de validation)", fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flat, sample_images):
    result = best_model.predict(img_path, conf=0.25, verbose=False)[0]
    img    = mpimg.imread(img_path)
    h, w   = img.shape[:2]

    ax.imshow(img)
    ax.axis('off')

    nb_objets = 0
    for box in result.boxes:
        conf    = float(box.conf[0])
        cls_idx = int(box.cls[0])
        cls_nom = result.names[cls_idx]
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        couleur = [c/255 for c in COULEURS_RGB.get(cls_nom, (255, 255, 255))]
        rect    = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=couleur, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-4, f"{cls_nom} {conf:.0%}",
                fontsize=7, color=couleur,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))
        nb_objets += 1

    ax.set_title(f"{Path(img_path).name[:20]} — {nb_objets} objets", fontsize=8)

plt.tight_layout()
plt.savefig('/content/predictions_yolov8s.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisation des détections sauvegardée")


In [ ]:
# ─── ANALYSE DES ERREURS — Matrice de confiance par classe ────────────────────
# On regarde quelles classes sont bien détectées et lesquelles posent problème

print("🔍 Calcul des métriques par classe...")
metrics_detail = best_model.val(data=DATA_YAML, verbose=False)

# Précision et rappel par classe
if hasattr(metrics_detail.box, 'ap_class_index'):
    classes_idx = metrics_detail.box.ap_class_index
    ap_per_cls  = metrics_detail.box.ap50       # mAP@0.5 par classe
    noms_classes = list(CLASS_MAP.keys())

    fig, ax = plt.subplots(figsize=(10, 5))
    cls_names = [noms_classes[i] for i in classes_idx]
    bars = ax.barh(cls_names, ap_per_cls, color='#3498db', alpha=0.85)

    for bar, val in zip(bars, ap_per_cls):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

    ax.set_xlim(0, 1.15)
    ax.set_xlabel('mAP@0.5')
    ax.set_title('Performance par classe — YOLOv8s (Conduite de Nuit)', fontweight='bold')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.6, label='Seuil 50%')
    ax.legend()
    plt.tight_layout()
    plt.savefig('/content/map_par_classe.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("\n📊 mAP@0.5 par classe :")
    for nom, val in zip(cls_names, ap_per_cls):
        etoiles = '⭐' * int(val * 5)
        print(f"   {nom:20s} : {val:.4f} {etoiles}")


##Sauvegarde sur Google Drive

In [ ]:
# ─── SAUVEGARDE DES MODÈLES ET GRAPHIQUES SUR GOOGLE DRIVE ───────────────────
save_dir = '/content/drive/MyDrive/PROJET_IA_CONDUITE_NUIT'
os.makedirs(save_dir, exist_ok=True)

# Modèles entraînés
shutil.copy(chemin_n, f'{save_dir}/yolov8n_best.pt')
shutil.copy(chemin_s, f'{save_dir}/yolov8s_best.pt')

# Graphiques
for fichier in ['exploration_bdd100k.png', 'comparaison_modeles.png',
                'courbes_entrainement.png', 'predictions_yolov8s.png',
                'map_par_classe.png', 'visualisation_labels.png']:
    src = f'/content/{fichier}'
    if os.path.exists(src):
        shutil.copy(src, f'{save_dir}/{fichier}')

print("✅ Sauvegarde terminée sur Google Drive !")
print(f"   Dossier : {save_dir}")
print("\n📁 Fichiers sauvegardés :")
for f in os.listdir(save_dir):
    taille = os.path.getsize(f'{save_dir}/{f}') // 1024
    print(f"   {f:40s} {taille:6d} Ko")


### Passons au test du Agent LLM avec le deuxième notebook